In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day14_Big_Data_Processing") \
    .getOrCreate()

print("Spark session created successfully!")
print("Spark version:", spark.version)

Spark session created successfully!
Spark version: 4.2.0


In [0]:
test_df = spark.range(1, 6)

display(test_df)

id
1
2
3
4
5


01_Generate_Large_Dataset

In [0]:
from pyspark.sql import functions as F

# Generate 1 million sales records
sales_df = (
    spark.range(1, 1000001)
    .select(
        F.col("id").alias("sales_id"),

        (F.col("id") % 10000 + 1).alias("customer_id"),

        F.when(F.col("id") % 5 == 0, "Electronics")
         .when(F.col("id") % 5 == 1, "Clothing")
         .when(F.col("id") % 5 == 2, "Furniture")
         .when(F.col("id") % 5 == 3, "Groceries")
         .otherwise("Sports").alias("category"),

        F.when(F.col("id") % 4 == 0, "Bengaluru")
         .when(F.col("id") % 4 == 1, "Hyderabad")
         .when(F.col("id") % 4 == 2, "Chennai")
         .otherwise("Mumbai").alias("region"),

        ((F.col("id") % 1000) + 100)
        .cast("double").alias("sales_amount"),

        F.when(F.col("id") % 10 == 0, "Cancelled")
         .otherwise("Completed").alias("status")
    )
)

print("Dataset generated successfully!")

Dataset generated successfully!


In [0]:
print("Total Records:", sales_df.count())

display(sales_df.limit(10))


Total Records: 1000000


sales_id,customer_id,category,region,sales_amount,status
1,2,Clothing,Hyderabad,101.0,Completed
2,3,Furniture,Chennai,102.0,Completed
3,4,Groceries,Mumbai,103.0,Completed
4,5,Sports,Bengaluru,104.0,Completed
5,6,Electronics,Hyderabad,105.0,Completed
6,7,Clothing,Chennai,106.0,Completed
7,8,Furniture,Mumbai,107.0,Completed
8,9,Groceries,Bengaluru,108.0,Completed
9,10,Sports,Hyderabad,109.0,Completed
10,11,Electronics,Chennai,110.0,Cancelled


02_Inspect_Dataset

In [0]:
# 1. Count total records
print("Total Records:", sales_df.count())

# 2. Display column names
print("Column Names:", sales_df.columns)

# 3. Display dataset schema
print("\nDataset Schema:")
sales_df.printSchema()

# 4. Display first 10 records
print("\nSample Records:")
display(sales_df.limit(10))

# 5. Check number of partitions
partition_count = (
    sales_df
    .select(F.spark_partition_id())
    .distinct()
    .count()
)

print("Number of Partitions:", partition_count)

Total Records: 1000000
Column Names: ['sales_id', 'customer_id', 'category', 'region', 'sales_amount', 'status']

Dataset Schema:
root
 |-- sales_id: long (nullable = false)
 |-- customer_id: long (nullable = true)
 |-- category: string (nullable = false)
 |-- region: string (nullable = false)
 |-- sales_amount: double (nullable = true)
 |-- status: string (nullable = false)


Sample Records:


sales_id,customer_id,category,region,sales_amount,status
1,2,Clothing,Hyderabad,101.0,Completed
2,3,Furniture,Chennai,102.0,Completed
3,4,Groceries,Mumbai,103.0,Completed
4,5,Sports,Bengaluru,104.0,Completed
5,6,Electronics,Hyderabad,105.0,Completed
6,7,Clothing,Chennai,106.0,Completed
7,8,Furniture,Mumbai,107.0,Completed
8,9,Groceries,Bengaluru,108.0,Completed
9,10,Sports,Hyderabad,109.0,Completed
10,11,Electronics,Chennai,110.0,Cancelled


Number of Partitions: 8


03_Filter_Transform_Data

In [0]:
from pyspark.sql import functions as F

# Step 1: Filter only completed sales
completed_sales = sales_df.filter(
    F.col("status") == "Completed"
)

# Step 2: Filter high-value sales (above 500)
high_value_sales = completed_sales.filter(
    F.col("sales_amount") > 500
)

# Step 3: Select required columns
result_df = high_value_sales.select(
    "sales_id",
    "customer_id",
    "category",
    "region",
    "sales_amount"
)

# Step 4: Display record counts
print("Total Sales:", sales_df.count())
print("Completed Sales:", completed_sales.count())
print("High Value Sales:", high_value_sales.count())

# Step 5: Display sample results
display(result_df.limit(20))

Total Sales: 1000000
Completed Sales: 900000
High Value Sales: 540000


sales_id,customer_id,category,region,sales_amount
401,402,Clothing,Hyderabad,501.0
402,403,Furniture,Chennai,502.0
403,404,Groceries,Mumbai,503.0
404,405,Sports,Bengaluru,504.0
405,406,Electronics,Hyderabad,505.0
406,407,Clothing,Chennai,506.0
407,408,Furniture,Mumbai,507.0
408,409,Groceries,Bengaluru,508.0
409,410,Sports,Hyderabad,509.0
411,412,Clothing,Mumbai,511.0


04_Sales_Aggregation

In [0]:
from pyspark.sql import functions as F

# Group completed sales by product category
category_summary = (
    completed_sales
    .groupBy("category")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("sales_amount").alias("total_sales"),
        F.avg("sales_amount").alias("average_sales"),
        F.max("sales_amount").alias("maximum_sale"),
        F.min("sales_amount").alias("minimum_sale")
    )
    .orderBy(F.desc("total_sales"))
)

# Display category-wise sales summary
display(category_summary)

category,total_orders,total_sales,average_sales,maximum_sale,minimum_sale
Sports,200000,1.203E8,601.5,1099.0,104.0
Groceries,200000,1.201E8,600.5,1098.0,103.0
Furniture,200000,1.199E8,599.5,1097.0,102.0
Clothing,200000,1.197E8,598.5,1096.0,101.0
Electronics,100000,6.0E7,600.0,1095.0,105.0


05_Regional_Analysis

In [0]:
from pyspark.sql import functions as F

# Group completed sales by region
regional_summary = (
    completed_sales
    .groupBy("region")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("sales_amount").alias("total_revenue"),
        F.avg("sales_amount").alias("average_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

# Display regional sales summary
display(regional_summary)

region,total_orders,total_revenue,average_revenue
Mumbai,250000,1.5025E8,601.0
Hyderabad,250000,1.4975E8,599.0
Chennai,200000,1.2E8,600.0
Bengaluru,200000,1.2E8,600.0


06_Window_Functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: Define window specification
window_spec = (
    Window
    .partitionBy("region")
    .orderBy(F.desc("sales_amount"))
)

# Step 2: Rank sales within each region
ranked_sales = completed_sales.withColumn(
    "sales_rank",
    F.dense_rank().over(window_spec)
)

# Step 3: Filter top 5 sales per region
top_sales = ranked_sales.filter(
    F.col("sales_rank") <= 5
)

# Step 4: Display ranked sales
display(
    top_sales.orderBy("region", "sales_rank")
)

sales_id,customer_id,category,region,sales_amount,status,sales_rank
682996,2997,Clothing,Bengaluru,1096.0,Completed,1
73996,3997,Clothing,Bengaluru,1096.0,Completed,1
182996,2997,Clothing,Bengaluru,1096.0,Completed,1
79996,9997,Clothing,Bengaluru,1096.0,Completed,1
817996,7997,Clothing,Bengaluru,1096.0,Completed,1
244996,4997,Clothing,Bengaluru,1096.0,Completed,1
391996,1997,Clothing,Bengaluru,1096.0,Completed,1
95996,5997,Clothing,Bengaluru,1096.0,Completed,1
585996,5997,Clothing,Bengaluru,1096.0,Completed,1
341996,1997,Clothing,Bengaluru,1096.0,Completed,1


07_Spark_Performance

In [0]:
from pyspark.sql import functions as F

# Step 1: Examine Spark execution plan
print("SPARK EXECUTION PLAN")

category_summary.explain()

# Step 2: Repartition dataset into 8 partitions
partitioned_sales = sales_df.repartition(8)

# Step 3: Identify partition distribution
partition_distribution = (
    partitioned_sales
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

# Step 4: Display partition distribution
print("PARTITION DISTRIBUTION")

display(partition_distribution)

SPARK EXECUTION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [total_sales#10426 DESC NULLS LAST]
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10028]
               +- PhotonShuffleExchangeSink rangepartitioning(total_sales#10426 DESC NULLS LAST, 16)
                  +- PhotonGroupingAgg(keys=[category#10382], functions=[finalmerge_count(merge count#10567L) AS count(1)#10559L, finalmerge_sum(merge sum#10569) AS sum(sales_amount)#10560, finalmerge_max(merge max#10571) AS max(sales_amount)#10562, finalmerge_min(merge min#10573) AS min(sales_amount)#10563])
                     +- PhotonShuffleExchangeSource false
                        +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10022]
                           +- PhotonShuffleExchangeSink hashpartitioning(category#10382, 16)
                              +- Phot

partition_id,count
0,125000
1,125000
2,125000
3,125000
4,125000
5,125000
6,125000
7,125000


08_Final_Processing_Report

In [0]:
from pyspark.sql import functions as F

# Step 1: Generate final processing report
final_report = completed_sales.agg(
    F.count("*").alias("Total_Completed_Orders"),
    F.sum("sales_amount").alias("Total_Revenue"),
    F.avg("sales_amount").alias("Average_Order_Value"),
    F.max("sales_amount").alias("Highest_Order_Value"),
    F.min("sales_amount").alias("Lowest_Order_Value")
)

# Step 2: Display final report
print("FINAL SALES PROCESSING REPORT")
display(final_report)

# Step 3: Display processing summary
print("===================================")
print("BIG DATA PROCESSING COMPLETED")
print("===================================")

print("Input Records:", sales_df.count())
print("Processed Records:", completed_sales.count())
print("High Value Records:", high_value_sales.count())

print(
    "Categories:",
    sales_df.select("category").distinct().count()
)

print(
    "Regions:",
    sales_df.select("region").distinct().count()
)

print("===================================")

FINAL SALES PROCESSING REPORT


Total_Completed_Orders,Total_Revenue,Average_Order_Value,Highest_Order_Value,Lowest_Order_Value
900000,5.4E8,600.0,1099.0,101.0


BIG DATA PROCESSING COMPLETED
Input Records: 1000000
Processed Records: 900000
High Value Records: 540000
Categories: 5
Regions: 4
